# 05 — Deep Learning: CNN & LSTM Autoencoders

Trains Conv1D and LSTM autoencoders for anomaly detection via reconstruction error.

**Requirements:** GPU accelerator enabled in Kaggle settings.

**Input:** `data/labeled/` from notebook 01
**Output:** `models/cnn_autoencoder.h5`, `models/lstm_autoencoder.h5`

In [ ]:
!pip install -q lightkurve astroquery PyWavelets matplotlib

import sys, os, glob

# Auto-detect codebase path
CODEBASE_PATH = None
for candidate in [
    '/kaggle/input/exopattern-codebase',
    *glob.glob('/kaggle/input/exopattern-codebase/*/'),
    *glob.glob('/kaggle/input/exopattern*/'),
    *glob.glob('/kaggle/input/exopattern*/*/'),
]:
    if os.path.isdir(os.path.join(candidate, 'backend')):
        CODEBASE_PATH = candidate.rstrip('/')
        break

if CODEBASE_PATH is None:
    raise FileNotFoundError(
        "Could not find 'backend/' directory in /kaggle/input/. "
        "Upload the project root (containing backend/ and experiments/) as a Kaggle dataset."
    )

sys.path.insert(0, CODEBASE_PATH)
print(f"Codebase found at: {CODEBASE_PATH}")

import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Check GPU
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 1. Load Dataset and Extract Features

In [ ]:
from backend.ml.preprocessing import LightCurvePreprocessor

# Auto-detect data directory
DATA_DIR = None
for candidate in [
    '/kaggle/input/exopattern-labeled-data/data/labeled',
    *glob.glob('/kaggle/input/exopattern-labeled*/data/labeled'),
    *glob.glob('/kaggle/input/exopattern-labeled*/*/data/labeled'),
    '/kaggle/working/data/labeled',
    'data/labeled',
]:
    if os.path.exists(os.path.join(candidate, 'metadata.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find labeled dataset (metadata.csv) in /kaggle/input/")

print(f"Using data from: {DATA_DIR}")

metadata = pd.read_csv(f'{DATA_DIR}/metadata.csv')
pp = LightCurvePreprocessor()
window_size = 50

all_features = []
all_labels = []

for _, row in metadata.iterrows():
    lc_path = f"{DATA_DIR}/lightcurves/{row['filename']}"
    if not os.path.exists(lc_path):
        continue
    df = pd.read_csv(lc_path)
    labels = df['label'].values
    df_proc = pp.preprocess(df, normalize=True)
    features = pp.extract_features(df_proc, window_size)
    
    stride = max(1, window_size // 4)
    window_labels = []
    for i in range(0, len(df_proc) - window_size + 1, stride):
        window_lab = labels[i:i + window_size]
        window_labels.append(int(window_lab.max() > 0))
    
    n = min(len(features), len(window_labels))
    all_features.append(features[:n])
    all_labels.append(np.array(window_labels[:n]))

X = np.vstack(all_features)
y = np.concatenate(all_labels)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Dataset: {X.shape[0]} windows, {X.shape[1]} features")
print(f"Anomalous: {y.sum()} ({y.mean()*100:.1f}%)")

# Train on normal data only (autoencoder approach)
X_normal = X[y == 0]
print(f"Normal training set: {X_normal.shape[0]} windows")

## 2. Train Conv1D Autoencoder

In [ ]:
from backend.ml.deep_models import Conv1DAutoencoder

cnn_ae = Conv1DAutoencoder(
    input_dim=X.shape[1],
    latent_dim=8,
    contamination=0.1,
    random_state=RANDOM_SEED,
)

# Fit on normal data
cnn_ae.fit(X_normal, epochs=100, batch_size=64, verbose=1)
print("Conv1D Autoencoder training complete.")

In [ ]:
# Plot training loss
if hasattr(cnn_ae, 'history_') and cnn_ae.history_ is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(cnn_ae.history_.history['loss'], label='Train Loss')
    if 'val_loss' in cnn_ae.history_.history:
        ax.plot(cnn_ae.history_.history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('Conv1D Autoencoder Training')
    ax.legend()
    plt.tight_layout()
    plt.savefig('/kaggle/working/cnn_ae_training.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# Evaluate on full dataset
cnn_preds = cnn_ae.predict(X)
cnn_scores = cnn_ae.score_samples(X)

cnn_anomaly = (cnn_preds == -1)
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

print(f"CNN AE Results:")
print(f"  Precision: {precision_score(y, cnn_anomaly, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y, cnn_anomaly, zero_division=0):.3f}")
print(f"  F1:        {f1_score(y, cnn_anomaly, zero_division=0):.3f}")
try:
    print(f"  ROC-AUC:   {roc_auc_score(y, -cnn_scores):.3f}")
except:
    print(f"  ROC-AUC:   N/A")

## 3. Train LSTM Autoencoder

In [ ]:
from backend.ml.deep_models import LSTMAutoencoder

lstm_ae = LSTMAutoencoder(
    input_dim=X.shape[1],
    seq_length=10,
    latent_dim=32,
    contamination=0.1,
    random_state=RANDOM_SEED,
)

# Fit on normal data
lstm_ae.fit(X_normal, epochs=50, batch_size=64, verbose=1)
print("LSTM Autoencoder training complete.")

In [ ]:
# Plot training loss
if hasattr(lstm_ae, 'history_') and lstm_ae.history_ is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(lstm_ae.history_.history['loss'], label='Train Loss')
    if 'val_loss' in lstm_ae.history_.history:
        ax.plot(lstm_ae.history_.history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('LSTM Autoencoder Training')
    ax.legend()
    plt.tight_layout()
    plt.savefig('/kaggle/working/lstm_ae_training.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# Evaluate on full dataset
lstm_preds = lstm_ae.predict(X)
lstm_scores = lstm_ae.score_samples(X)

lstm_anomaly = (lstm_preds == -1)

print(f"LSTM AE Results:")
print(f"  Precision: {precision_score(y, lstm_anomaly, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y, lstm_anomaly, zero_division=0):.3f}")
print(f"  F1:        {f1_score(y, lstm_anomaly, zero_division=0):.3f}")
try:
    print(f"  ROC-AUC:   {roc_auc_score(y, -lstm_scores):.3f}")
except:
    print(f"  ROC-AUC:   N/A")

## 4. Save Models

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save CNN AE
if cnn_ae.model is not None:
    cnn_ae.model.save(OUTPUT_DIR / 'cnn_autoencoder.h5')
    print(f"Saved CNN AE to {OUTPUT_DIR / 'cnn_autoencoder.h5'}")
    with open(OUTPUT_DIR / 'cnn_autoencoder_meta.json', 'w') as f:
        json.dump({
            'threshold': float(cnn_ae.threshold) if cnn_ae.threshold is not None else None,
            'latent_dim': cnn_ae.latent_dim,
        }, f)

# Save LSTM AE
if lstm_ae.model is not None:
    lstm_ae.model.save(OUTPUT_DIR / 'lstm_autoencoder.h5')
    print(f"Saved LSTM AE to {OUTPUT_DIR / 'lstm_autoencoder.h5'}")
    with open(OUTPUT_DIR / 'lstm_autoencoder_meta.json', 'w') as f:
        json.dump({
            'threshold': float(lstm_ae.threshold) if lstm_ae.threshold is not None else None,
            'seq_length': lstm_ae.seq_length,
            'latent_dim': lstm_ae.latent_dim,
        }, f)

print("\nAll deep learning models saved.")

## 5. Reconstruction Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CNN AE
ax = axes[0]
ax.hist(-cnn_scores[y == 0], bins=50, alpha=0.5, label='Normal', density=True)
ax.hist(-cnn_scores[y == 1], bins=50, alpha=0.5, label='Transit', density=True)
if cnn_ae.threshold is not None:
    ax.axvline(cnn_ae.threshold, color='red', linestyle='--', label='Threshold')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_title('Conv1D Autoencoder')
ax.legend()

# LSTM AE
ax = axes[1]
ax.hist(-lstm_scores[y == 0], bins=50, alpha=0.5, label='Normal', density=True)
ax.hist(-lstm_scores[y == 1], bins=50, alpha=0.5, label='Transit', density=True)
if lstm_ae.threshold is not None:
    ax.axvline(lstm_ae.threshold, color='red', linestyle='--', label='Threshold')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_title('LSTM Autoencoder')
ax.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/reconstruction_errors.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# List output files
print("Output files:")
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")

## Done!

Download `models/` from this notebook's output. Place `.h5` files in `artifacts/models/` locally.

Combined with notebook 03's `.pkl` files, you now have all trained models for the Streamlit frontend.

## 6. Package Models as ZIP

In [ ]:
import shutil

# Zip deep learning models (.h5 + metadata JSON)
shutil.make_archive('/kaggle/working/exopattern-dl-models', 'zip', '/kaggle/working/models')
zip_size = os.path.getsize('/kaggle/working/exopattern-dl-models.zip') / (1024 * 1024)
print(f"Created exopattern-dl-models.zip ({zip_size:.1f} MB)")
print("Download and place contents in artifacts/models/ locally.")
print("Contains: cnn_autoencoder.h5, lstm_autoencoder.h5, *_meta.json")